In [3]:
!pip uninstall -y sentence-transformers huggingface-hub InstructorEmbedding transformers
!pip install sentence-transformers==2.2.2 huggingface_hub==0.20.3 InstructorEmbedding
# after this cell finishes, go to runtime and restart the session
# DO NOT run anything else before restarting

Found existing installation: sentence-transformers 5.6.0
Uninstalling sentence-transformers-5.6.0:
  Successfully uninstalled sentence-transformers-5.6.0
Found existing installation: huggingface_hub 1.20.1
Uninstalling huggingface_hub-1.20.1:
  Successfully uninstalled huggingface_hub-1.20.1
Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu!!")

CUDA available: True
GPU: Tesla T4


## upload

In [ ]:
from google.colab import files
uploaded = files.upload()   # processed_documents.zip

import zipfile
with zipfile.ZipFile("processed_documents_recursive.zip") as z:
    z.extractall("data")

import json
from pathlib import Path
PROCESSED = Path("data/processed_documents_recursive")

def load_corpus():
    master = json.loads((PROCESSED / "metadata.json").read_text(encoding="utf-8"))
    corpus, skipped = [], 0
    for fname, meta in master.items():
        if meta.get("is_duplicate"):
            skipped += 1
            continue
        stem = Path(fname).stem
        dj = PROCESSED / f"{stem}.json"
        if not dj.exists():
            continue
        data = json.loads(dj.read_text(encoding="utf-8"))
        for c in data["chunks"]["chunks"]:
            t = c["text"].strip()
            if not t:
                continue
            corpus.append({"chunk_id": f"{stem}__{c['chunk_index']}",
                           "text": t, "source_doc": fname,
                           "language": meta.get("primary_language"),
                           "token_count": c.get("token_count")})
    print(f"[loader] {len(corpus)} chunks | {skipped} dupe docs skipped")
    return corpus

corpus = load_corpus()
assert len(corpus) == 10456, f"EXPECTED 10456, GOT {len(corpus)} — corpus mismatch!"

Saving processed_documents_recursive.zip to processed_documents_recursive.zip
[loader] 7325 chunks | 10 dupe docs skipped


AssertionError: EXPECTED 10456, GOT 7325 — corpus mismatch!

In [ ]:
!ls

data  processed_documents_fixed.zip  sample_data


## embed
official Instructor API requires a prompt, allows task-specific instructions

In [ ]:
import numpy as np, json, time
from InstructorEmbedding import INSTRUCTOR
from google.colab import files

model = INSTRUCTOR("hkunlp/instructor-xl")   # ~4.9GB download, first load is slow

INSTRUCTION = "Represent the document for retrieval:"
# Official API: list of [instruction, text] PAIRS, not plain strings
pairs = [[INSTRUCTION, c["text"]] for c in corpus]
ids   = [c["chunk_id"] for c in corpus]

t = time.time()
vecs = model.encode(pairs, batch_size=32, show_progress_bar=True,
                    normalize_embeddings=True)   # normalize to match the other 4
dt = time.time() - t
vecs = np.asarray(vecs, dtype=np.float32)

np.save("instructor-xl.npy", vecs)
with open("instructor-xl.meta.json", "w") as f:
    json.dump({"model": "instructor-xl", "hf_id": "hkunlp/instructor-xl",
               "instruction": INSTRUCTION, "dim": int(vecs.shape[1]),
               "count": int(vecs.shape[0]), "chunk_ids": ids,
               "encode_seconds": round(dt, 1)}, f)

print(f"done: {vecs.shape} in {dt:.1f}s ({1000*dt/len(corpus):.0f} ms/chunk)")
files.download("instructor-xl.npy")
files.download("instructor-xl.meta.json")

load INSTRUCTOR_Transformer
max_seq_length  512


Batches:   0%|          | 0/229 [00:00<?, ?it/s]

done: (7325, 768) in 3680.5s (502 ms/chunk)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## embedding queries

In [3]:
from google.colab import files
up = files.upload()   # pick eval_queries.json

Saving eval_queries.json to eval_queries.json


In [4]:
import json, numpy as np
from InstructorEmbedding import INSTRUCTOR
from google.colab import files

queries = json.loads(open("eval_queries.json", encoding="utf-8").read())["queries"]

# QUERY-side instruction — differs from the document one you used for the corpus
Q_INSTRUCTION = "Represent the question for retrieving supporting documents:"
pairs = [[Q_INSTRUCTION, q["query"]] for q in queries]

model = INSTRUCTOR("hkunlp/instructor-xl")
vecs = np.asarray(model.encode(pairs, normalize_embeddings=True), dtype=np.float32)
print("shape:", vecs.shape)   # must be (45, 768)

np.save("instructor-xl.npy", vecs)
files.download("instructor-xl.npy")

load INSTRUCTOR_Transformer
max_seq_length  512
shape: (45, 768)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>